# Pré-processamento das Bases da SSP-SP
O objetivo é consolidar as fontes obtidas pela transparência da **Secretária de Segurança Pública de São Paulo (SPP-SP)** em um único **dataset**.

## 1. Bibliotecas e Configuração Inicial

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

***Arquivos***

### 1.1 Fontes de Dados

São listados os arquivos brutos (`.xlsx`) disponibilizados pela transparência da SSP-SP, um por ano, de **2022 a 2026**. Cada arquivo contém os Boletins de Ocorrência (B.O.) registrados no respectivo período.

In [2]:
files = ['datasets/SPDadosCriminais_2022.xlsx',
         'datasets/SPDadosCriminais_2023.xlsx',
         'datasets/SPDadosCriminais_2024.xlsx',
         'datasets/SPDadosCriminais_2025.xlsx',
         'datasets/SPDadosCriminais_2026.xlsx']

In [3]:
list_dataframes = []

### 1.2 Leitura e Consolidação das Planilhas

Cada arquivo Excel possui **múltiplas abas** (uma por semestre). O código:

1. Lê **todas as abas** de cada arquivo (`sheet_name=None`), retornando um dicionário `{nome_da_aba: dataframe}`;
2. **Ignora a primeira aba** (`abas[1:]`), que corresponde a uma folha de rosto/legenda, e não a dados propriamente ditos;
3. Adiciona duas colunas de rastreabilidade em cada dataframe — `origin_file` e `origin_aba` — para manter o registro de **qual arquivo e aba deram origem a cada linha**;
4. Acumula todos os dataframes resultantes em `list_dataframes`.

In [4]:
for file in files:
    abas_dict = pd.read_excel(file, sheet_name=None)
    abas = list(abas_dict.items())
    for name_aba, df_aba in abas[1:]:
        df_aba['origin_file'] = file
        df_aba['origin_aba'] = name_aba
        list_dataframes.append(df_aba)

### 1.3 Concatenação da Base Bruta

Todos os dataframes (de todos os anos e abas) são concatenados verticalmente em um único dataframe (`df_raw`), consolidando os B.O. de 2022 a 2026 em uma única estrutura de dados.

In [12]:
df_raw = pd.concat(list_dataframes, ignore_index=True)

In [7]:
df_raw.head()

,NOME_DEPARTAMENTO,NOME_SECCIONAL,NOME_DELEGACIA,CIDADE,NUM_BO,ANO_BO,DATA_COMUNICACAO_BO,DATA_OCORRENCIA_BO,HORA_OCORRENCIA_BO,DESCR_PERIODO,...,origin_aba,NOME_MUNICIPIO,DATA_REGISTRO,DESC_PERIODO,DESCR_TIPOLOCAL,NOME_DELEGACIA_CIRCUNSCRICAO,NOME_DEPARTAMENTO_CIRCUNSCRICAO,NOME_SECCIONAL_CIRCUNSCRICAO,NOME_MUNICIPIO_CIRCUNSCRICAO,COD IBGE
0,DECAP,DEL.SEC.8º SAO MATEUS,08ª DDM SAO MATEUS,S.PAULO,AC0851,2022,2022-01-13 00:00:00,2021-09-01 00:00:00,NaN,EM HORA INCERTA,...,JAN-JUN_2022,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DECAP,DEL.SEC.8º SAO MATEUS,54º D.P. CID. TIRADENTES,S.PAULO,4,2022,2022-01-03 00:00:00,2021-12-31 00:00:00,15:20:00,NaN,...,JAN-JUN_2022,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,DECAP,DEL.SEC.5º LESTE,29º D.P. VILA PRUDENTE,S.PAULO,7,2022,2022-01-04 00:00:00,2021-12-30 00:00:00,NaN,EM HORA INCERTA,...,JAN-JUN_2022,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,DECAP,DEL.SEC.1º CENTRO,01º D.P. SE,S.PAULO,8,2022,2022-01-03 00:00:00,2022-01-03 00:00:00,16:00:00,NaN,...,JAN-JUN_2022,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DECAP,DEL.SEC.1º CENTRO,06º D.P. CAMBUCI,S.PAULO,8,2022,2022-01-03 00:00:00,2022-01-03 00:00:00,13:30:00,NaN,...,JAN-JUN_2022,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


***Salvar***

In [13]:
df_raw.to_csv('datasets/SPDadosCriminais_2022_2026.csv', index=False)

***Agregação***
<p>É necessário gerar a quantidade de ocorrências por tipo de crime, localização, quinzena, mês e ano para subsidiar o cálculo do risco de <b>criminalidade</b>.</p>

In [3]:
df_raw = pd.read_csv('datasets/SPDadosCriminais_2022_2026.csv')

C:\Users\Usuario\AppData\Local\Temp\ipykernel_24656\1190081205.py:1: DtypeWarning: Columns (3,6,9,13,14,15,16,17,18,19,31,32,33,34,35,36,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('datasets/SPDadosCriminais_2022_2026.csv')


## 2. Padronização da Coluna de Cidade

A base possui duas colunas que podem indicar o município do B.O.: `NOME_MUNICIPIO_CIRCUNCRIÇÃO` (município da circunscrição/delegacia responsável [local onde ocorreu o fato]) e `NOME_MUNICIPIO` (município registrado). Como a primeira apresenta uma quantidade relevante de nulos, é necessário avaliar se a segunda pode servir de **fallback**, e então consolidar as duas em uma única coluna `city`.

In [4]:
len(df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]) / len(df_raw)

0.10384811135788

In [5]:
city_null = df_raw[df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].isnull()]

In [6]:
len(city_null[city_null['NOME_MUNICIPIO'].isnull()]) / len(city_null)

0.0

Confirmada a viabilidade do fallback, a coluna `city` é criada a partir de `NOME_MUNICIPIO_CIRCUNCRIÇÃO`, preenchendo os valores nulos com `NOME_MUNICIPIO`.

In [7]:
df_raw['city'] = df_raw['NOME_MUNICIPIO_CIRCUNCRIÇÃO'].fillna(df_raw['NOME_MUNICIPIO'])

In [8]:
df_raw['city'] = df_raw['city'].str.strip()

In [9]:
len(df_raw[df_raw['city'].isnull()]) / len(df_raw)

0.0

## 3. Filtragem por Tipo de Crime

O estudo tem como foco **Roubos e Furtos**. Antes de aplicar qualquer filtro, é feito um levantamento dos valores únicos da coluna `NATUREZA_APURADA`, para identificar todas as categorias existentes e decidir quais devem ser mantidas.

In [10]:
df_raw['NATUREZA_APURADA'].unique()

array(['ESTUPRO', 'FURTO - OUTROS', 'FURTO DE VEÍCULO',
       'HOMICÍDIO DOLOSO',
       'LESÃO CORPORAL CULPOSA POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL DOLOSA', 'ROUBO - OUTROS', 'ROUBO DE CARGA',
       'ROUBO DE VEÍCULO', 'TENTATIVA DE HOMICIDIO',
       'TRAFICO DE ENTORPECENTES', 'LESÃO CORPORAL CULPOSA – OUTRAS',
       'ESTUPRO DE VULNERÁVEL', 'FURTO DE CARGA',
       'HOMICIDIO CULPOSO POR ACIDENTE DE TRANSITO',
       'HOMICIDIO CULPOSO OUTROS', 'EXTORSÃO MEDIANTE SEQUESTRO',
       'LATROCÍNIO', 'ROUBO A BANCO', 'LESÃO CORPORAL SEGUIDA DE MORTE',
       'HOMICÍDIO DOLOSO POR ACIDENTE DE TRÂNSITO',
       'HOMICÍDIO CULPOSO POR ACIDENTE DE TRÂNSITO',
       'LESÃO CORPORAL CULPOSA - OUTRAS', 'PORTE DE ARMA',
       'PORTE DE ENTORPECENTES', 'TRÁFICO DE ENTORPECENTES',
       'APREENSÃO DE ENTORPECENTES', 'TENTATIVA DE HOMICÍDIO',
       'HOMICÍDIO CULPOSO OUTROS', 'Furto de Veículo', 'Roubo de Veículo',
       'Furto - Outros', 'Estupro', 'Estupro de Vulnerável',
  

Como o objetivo do estudo é analisar **Roubos e Furtos**, serão mantidas as seguintes categorias:
- 'FURTO - OUTROS'
- 'FURTO DE VEÍCULO'
- 'FURTO DE CARGA'
- 'ROUBO - OUTROS'
- 'ROUBO DE CARGA'
- 'ROUBO DE VEÍCULO'
- 'ROUBO A BANCO'
<p>Em seguida, <b>'ROUBO A BANCO'</b> será incorporado em <b>'ROUBO - OUTROS'</b></p>

In [11]:
list_crimes = ['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
               'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA',
                'ROUBO A BANCO', 'Furto de Veículo', 'Roubo de Veículo',
                    'Furto - Outros']

In [12]:
df_raw = df_raw[df_raw['NATUREZA_APURADA'].isin(list_crimes)]

In [13]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].str.upper()

In [14]:
df_raw['NATUREZA_APURADA'] = df_raw['NATUREZA_APURADA'].map(lambda x: 'ROUBO - OUTROS' if x == 'ROUBO A BANCO' else x)

In [15]:
df_raw['NATUREZA_APURADA'].unique()

array(['FURTO - OUTROS', 'FURTO DE VEÍCULO', 'ROUBO - OUTROS',
       'ROUBO DE CARGA', 'ROUBO DE VEÍCULO', 'FURTO DE CARGA'],
      dtype=object)

Será mantido apenas B.O. de munícipios de Região Metropolitana de São Paulo. A lista de munícipios está disponível em: https://pt.wikipedia.org/wiki/Regi%C3%A3o_Metropolitana_de_S%C3%A3o_Paulo

## 4. Filtragem Geográfica: Região Metropolitana de São Paulo

O escopo do estudo é restrito aos municípios da **Região Metropolitana de São Paulo (RMSP)**. A lista de municípios é definida manualmente com base na composição oficial da RMSP, e utilizada para filtrar a base pela coluna `city`.

In [16]:
list_city = ['ARUJA', 'BARUERI', 'BIRITIBA-MIRIM', 'CAIEIRAS', 'CAJAMAR',
             'CARAPICUIBA', 'COTIA', 'DIADEMA', 'EMBU DAS ARTES', 'EMBU-GUACU',
                'FERRAZ DE VASCONCELOS', 'FRANCISCO MORATO', 'FRANCO DA ROCHA', 'GUARAREMA',
                    'GUARULHOS', 'ITAPECERICA DA SERRA', 'ITAPEVI', 'ITAQUAQUECETUBA',
                        'JANDIRA', 'JUQUITIBA', 'MAIRIPORA', 'MAUA', 'MOGI DAS CRUZES', 'OSASCO',
                        'PIRAPORA BOM JESUS', 'POA', 'RIBEIRAO PIRES', 'RIO GRANDE DA SERRA',
                            'SALESOPOLIS', 'SANTANA DE PARNAIBA', 'S.ANDRE', 'S.BERNADO DO CAMPO', 'S.CAETANO DO SUL', 'S.LOURENCO DA SERRA', 'S.PAULO', 'SUZANO', 'TABOAO DA SERRA', 'VARGEM GRANDE PAULISTA']

In [17]:
df_raw = df_raw[df_raw['city'].isin(list_city)]

## 5. Tratamento das Datas de Ocorrência

Antes de derivar mês, dia e ano do B.O., é avaliada a completude da coluna `DATA_OCORRENCIA_BO` e a possibilidade de usar `DATA_REGISTRO` como **fallback** para os poucos casos em que a data de ocorrência está ausente.

In [18]:
len(df_raw[df_raw['DATA_OCORRENCIA_BO'].isnull()]) / len(df_raw)

0.0001704398175820508

In [19]:
len(df_raw[df_raw['DATA_REGISTRO'].isnull() & df_raw['DATA_OCORRENCIA_BO'].isnull()]) / len(df_raw)

0.0

Ambas as colunas de data (`DATA_OCORRENCIA_BO` e `DATA_REGISTRO`) são convertidas para o tipo `datetime`, com `errors='coerce'` transformando qualquer valor inválido em nulo (`NaT`), em vez de interromper a execução.

In [20]:
df_raw['DATA_OCORRENCIA_BO'] = pd.to_datetime(df_raw['DATA_OCORRENCIA_BO'], errors='coerce')

In [21]:
df_raw['DATA_REGISTRO'] = pd.to_datetime(df_raw['DATA_REGISTRO'], errors='coerce')

### 6.1 Derivando Mês e Dia com Fallback

As colunas `MES_BO` e `DIA_BO` são extraídas de `DATA_OCORRENCIA_BO`. Quando essa data é nula, o mês/dia são preenchidos a partir de `DATA_REGISTRO` (fallback), reduzindo a perda de registros por ausência de uma única fonte de data.

In [22]:
df_raw['MES_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.month.astype('Int64').fillna(df_raw['DATA_REGISTRO'].dt.month.astype('Int64'))
df_raw['DIA_BO'] = df_raw['DATA_OCORRENCIA_BO'].dt.day.astype('Int64').fillna(df_raw['DATA_REGISTRO'].dt.day.astype('Int64'))

In [23]:
len(df_raw[df_raw['MES_BO'].isnull() | df_raw['DIA_BO'].isnull()])

39

Os registros que permaneceram sem `MES_BO` e/ou `DIA_BO` — mesmo após o fallback — são descartados, por representarem um volume muito baixo (39 B.O.) e não terem nenhuma fonte de data válida.

In [24]:
df_raw = df_raw[df_raw['MES_BO'].notnull() & df_raw['DIA_BO'].notnull()]

### 6.2 Construção da Chave Temporal `DATA` (Quinzena)

Para a granularidade temporal da análise, é definida uma função que classifica cada registro em **1ª ou 2ª quinzena** do mês (dia ≤ 15 ou > 15), compondo uma chave no formato `quinzena-mês-ano`. Essa chave é aplicada linha a linha para gerar a coluna `DATA`, que será usada posteriormente como uma das dimensões de agregação.

In [25]:
def get_date(row):
    quinzena = 1 if row['DIA_BO'] <= 15 else 2
    return f"{quinzena}-{row['MES_BO']}-{row['ANO_BO']}"

In [26]:
df_raw['DATA'] = df_raw.apply(get_date, axis=1)

In [27]:
len(df_raw[df_raw['LATITUDE'].isnull()]) / len(df_raw)

0.11340103685817769

## 7. Tratamento de Registros Duplicados
Será avaliado a presença de registros duplicados e seus impactos

In [28]:
df_raw.duplicated(subset=['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'NATUREZA_APURADA']).sum()

644210

In [29]:
df_raw.duplicated(subset=['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'NATUREZA_APURADA']).sum() / len(df_raw) * 100

25.416834347308036

Cerca de **25%** da base são **registros duplicados**. Portanto, como um dado duplicado não agrega informação, logo serão removidos da base.

In [35]:
df_raw = df_raw.drop_duplicates(subset=['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'NATUREZA_APURADA'], keep='first')

## 8. Tratamento de Coordenadas Geográficas

Antes de qualquer correção, é avaliada a proporção de valores nulos em `LATITUDE` e `LONGITUDE`, individualmente e em conjunto, para dimensionar o problema e orientar a estratégia de imputação.

In [37]:
len(df_raw[df_raw['LONGITUDE'].isnull()]) / len(df_raw)

0.005028645185863085

In [38]:
len(df_raw[df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull()])

9506

Os registros referentes ao ano de **2021** são descartados neste ponto.

In [39]:
df_raw = df_raw[df_raw['ANO_BO'] != 2021]

### 8.1 Identificando Coordenadas Inválidas

É criada a coluna `COORD_IMPUTADA`, que sinaliza registros cuja coordenada precisa ser tratada/imputada. Um registro é marcado como inválido quando:

- `LATITUDE` ou `LONGITUDE` são nulos; **ou**
- o valor (como string) não começa com `-` (coordenadas no Brasil devem ser negativas); **ou**
- o valor é literalmente o caractere `'-'` (sem dígitos).

In [40]:
df_raw['COORD_IMPUTADA'] = df_raw['LATITUDE'].isnull() | df_raw['LONGITUDE'].isnull() | (~df_raw['LATITUDE'].astype(str).str.match(r'^-')) | (~df_raw['LONGITUDE'].astype(str).str.match(r'^-')) | (df_raw['LATITUDE'] == '-') | (df_raw['LONGITUDE'] == '-')

### 8.2 Funções de Correção de Coordenadas

São definidas três funções auxiliares:

- **`fix_coord_lat` / `fix_coord_long`**: zeram a coordenada de registros marcados como `COORD_IMPUTADA`, preparando-os para receberem o valor definitivo mais adiante;
- **`fix_coord`**: corrige erros de **escala** — coordenadas gravadas sem o ponto decimal (ex: `-4664112.0` em vez de `-46.64112`) — dividindo o valor por 10 repetidamente até que seu módulo fique dentro do limite geográfico válido (90° para latitude, 180° para longitude).

In [41]:
def fix_coord_lat(row):
    return 0 if row['COORD_IMPUTADA'] else row['LATITUDE']

def fix_coord_long(row):
    return 0 if row['COORD_IMPUTADA'] else row['LONGITUDE']

def fix_coord(value, limit=90):
    fixed_value = value
    while abs(fixed_value) > limit:
        fixed_value /= 10
    return fixed_value

Substitui valores **inválidos** por *0*

In [42]:
df_raw['LONGITUDE'] = df_raw.apply(fix_coord_long, axis=1)
df_raw['LATITUDE'] = df_raw.apply(fix_coord_lat, axis=1)

In [43]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].astype('Float64')
df_raw['LATITUDE'] = df_raw['LATITUDE'].astype('Float64')

**Corrigi** as coordenadas para um **intervalo válido**

In [44]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].map(fix_coord)
df_raw['LATITUDE'] = df_raw['LATITUDE'].map(fix_coord)

In [45]:
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].replace(0, np.nan)
df_raw['LATITUDE'] = df_raw['LATITUDE'].replace(0, np.nan)

### 8.3 Imputação por Centróide da Cidade

Para os registros sem coordenada válida, a estratégia adotada é preencher com a **mediana** de latitude/longitude dos demais registros da mesma cidade (`city`). A mediana é usada em vez da média por ser mais robusta a eventuais outliers remanescentes na base.

In [46]:
median_point_city = (
    df_raw.dropna(subset=['LATITUDE', 'LONGITUDE'])
    .groupby('city')[['LATITUDE', 'LONGITUDE']]
    .median()
)

In [47]:
df_raw['LATITUDE'] = df_raw['LATITUDE'].fillna(df_raw['city'].map(median_point_city['LATITUDE']))
df_raw['LONGITUDE'] = df_raw['LONGITUDE'].fillna(df_raw['city'].map(median_point_city['LONGITUDE']))

## 9. Agregação Final

Com a base limpa e padronizada, o último passo é agregá-la por **latitude, longitude e data (quinzena)**, gerando as métricas que servirão de base para o cálculo do risco de criminalidade:

- `cnt_total`: contagem **total** de ocorrências por combinação de latitude/longitude/data;
- `cnt_by_type`: contagem de ocorrências **por tipo de crime** (`NATUREZA_APURADA`), uma coluna por categoria, para a mesma combinação de chaves.

In [48]:
cnt_total = (
    df_raw.groupby(['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'MES_BO', 'ANO_BO'])
        .size()
        .reset_index(name='cnt_crime')
)

In [49]:
cnt_by_type = (
    df_raw.pivot_table(
        index=['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'MES_BO', 'ANO_BO'],
        columns='NATUREZA_APURADA',
        aggfunc='size',
        fill_value=0
    ).reset_index()
)

As duas agregações (`cnt_total` e `cnt_by_type`) são unidas em um único dataframe final (`df_group`), contendo a contagem total e a contagem por tipo de crime para cada combinação de localização e data.

In [50]:
df_group = cnt_total.merge(cnt_by_type, on=['LATITUDE', 'LONGITUDE', 'DATA', 'city', 'MES_BO', 'ANO_BO'])

In [51]:
df_group

,LATITUDE,LONGITUDE,DATA,city,MES_BO,ANO_BO,cnt_crime,FURTO - OUTROS,FURTO DE CARGA,FURTO DE VEÍCULO,ROUBO - OUTROS,ROUBO DE CARGA,ROUBO DE VEÍCULO
0,-28.304600,-54.271865,1-12-2024,GUARULHOS,12,2024,1,1,0,0,0,0,0
1,-24.972870,-48.402852,2-10-2025,TABOAO DA SERRA,10,2025,1,0,0,0,0,0,1
2,-24.972870,-48.402852,2-11-2025,EMBU DAS ARTES,11,2025,1,0,0,0,0,0,1
3,-24.969331,-48.361020,2-2-2024,ITAPECERICA DA SERRA,2,2024,1,0,0,0,1,0,0
4,-24.969324,-48.361338,1-7-2023,JUQUITIBA,7,2023,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1800201,-21.039556,-49.930865,1-2-2023,S.PAULO,2,2023,1,1,0,0,0,0,0
1800202,-20.325371,-48.310855,1-6-2026,S.PAULO,6,2026,1,1,0,0,0,0,0
1800203,-19.882014,-43.896489,1-4-2025,S.PAULO,4,2025,1,0,0,0,1,0,0
1800204,-15.477469,-44.371940,2-7-2025,S.PAULO,7,2025,1,0,0,0,1,0,0
